# Fraud Detection Pipeline — End-to-EndThis notebook runs the complete workflow: data loading, outlier removal, feature engineering, model comparison, ensemble training with proper CV, and SHAP explainability.Click **Run All** to reproduce the full pipeline.

In [ ]:
import sys, os, warnings, timewarnings.filterwarnings('ignore')import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as snssns.set_style('whitegrid')sys.path.insert(0, '..')from src.data_processing import create_fraud_datasetfrom src.data_processing.outlier_handler import remove_fraud_outliers, get_outlier_summaryfrom src.data_processing.feature_engineering import AdvancedFeatureEngineeringfrom src.models.fraud_detector import (    LogisticRegressionDetector, RandomForestDetector, XGBoostDetector,    LightGBMDetector, AdaBoostDetector, EnsembleFraudDetector,)from src.models.explainer import FraudExplainerprint('All imports OK')

## 1. Load DataCredit Card Fraud Detection dataset (Kaggle). 284,807 transactions, 0.17% fraud rate.

In [ ]:
df = create_fraud_dataset(n_samples=30000)print(f'{len(df):,} rows, fraud rate: {df["Class"].mean():.4%}')print(f'Fraud cases: {int(df["Class"].sum())}')df.describe()

## 2. Outlier RemovalIQR-based on V14, V12, V10 (strongest fraud-correlated features). Thresholds computed on a balanced subsample.

In [ ]:
df_clean = remove_fraud_outliers(df, multiplier=1.5)print(f'After cleaning: {len(df_clean):,} rows, fraud rate: {df_clean["Class"].mean():.4%}')

## 3. Feature EngineeringOnline/offline feature split: 50+ features from 31 raw columns. Velocity, anomaly, clustering, interaction features.

In [ ]:
fe = AdvancedFeatureEngineering(target_column='Class')df_eng = fe.fit_transform(df_clean)X, y = df_eng.drop('Class', axis=1), df_eng['Class']print(f'Features: {X.shape[1]} (from {df.shape[1]} raw)')

## 4. Model ComparisonTrain all 5 models on the same split. Compare AUC, F1, recall.

In [ ]:
from sklearn.model_selection import train_test_splitX_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)models = {    'Logistic Regression': LogisticRegressionDetector(),    'Random Forest': RandomForestDetector(),    'XGBoost': XGBoostDetector(),    'LightGBM': LightGBMDetector(),    'AdaBoost': AdaBoostDetector(),}results = []for name, detector in models.items():    detector.train(X_tr, y_tr)    m = detector.evaluate_model(X_te, y_te)    results.append({'Model': name, **{k: round(v, 4) for k, v in m.items()}})comparison = pd.DataFrame(results).sort_values('f1_score', ascending=False)comparison[['Model', 'auc', 'f1_score', 'recall', 'precision']]

## 5. Ensemble Training with CV5-fold StratifiedKFold with SMOTE inside each fold — no data leakage. AUC-weighted soft voting.

In [ ]:
ensemble = EnsembleFraudDetector(random_state=42)t0 = time.time()ensemble.train(X, y, test_size=0.2, balance_data=True, use_cv=True, n_folds=5)print(f'\nTraining time: {time.time() - t0:.0f}s')print(f'Ensemble weights: {ensemble._model_weights}')pd.DataFrame([ensemble.ensemble_metrics]).T.rename(columns={0: 'value'})

## 6. Threshold OptimizationFind the threshold that maximizes F1 (not default 0.5).

In [ ]:
best_t = ensemble.optimize_threshold(X, y, metric='f1')

## 7. SHAP ExplainabilityWeighted multi-model SHAP. Top-5 risk factors per prediction.

In [ ]:
explainer = FraudExplainer(ensemble)explainer.fit_background(X.sample(200, random_state=42))sample_tx = X.head(1)result = explainer.explain_prediction(sample_tx)factors = result['per_prediction'][0]['top_risk_factors']print('Top-5 risk factors for sample transaction:')for f in factors:    direction = '+FRAUD' if f['direction'] == 'fraud' else '-normal'    print(f'  {f["feature"]:30s} SHAP={f["impact"]:+.6f} {direction}')# Global feature importancetop_features = list(result['global_importance'].items())[:10]fig, ax = plt.subplots(figsize=(10, 5))ax.barh([f[0] for f in reversed(top_features)], [f[1] for f in reversed(top_features)])ax.set_title('Global Feature Importance (weighted mean |SHAP|)')plt.tight_layout()plt.show()

## 8. SummaryModel comparison + ensemble metrics + SHAP top features.

In [ ]:
print('=' * 60)print('FRAUD DETECTION PIPELINE COMPLETE')print('=' * 60)print(f'Dataset: {len(df):,} transactions, {int(df["Class"].sum())} frauds')print(f'Features: {X.shape[1]} engineered from {df.shape[1]} raw')print(f'Ensemble F1: {ensemble.ensemble_metrics["f1_score"]:.4f}')print(f'Ensemble AUC: {ensemble.ensemble_metrics["roc_auc"]:.4f}')print(f'Optimal threshold: {best_t:.3f}')print(f'Top SHAP features: {[f["feature"] for f in factors[:3]]}')